In [15]:
import pandas as pd
from pathlib import Path

# Resolve path whether notebook runs in RAG_System or project root
candidates = [
    Path.cwd() / "Data" / "Final_dataset_ERCOT_v2.csv",
    Path.cwd().parent / "Data" / "Final_dataset_ERCOT_v2.csv",
]
for p in candidates:
    if p.exists():
        data_path = p
        break
else:
    raise FileNotFoundError("Final_dataset_ERCOT_v2.csv not found. Checked project and parent Data folders.")

# Load and index by timestamp
df = pd.read_csv(data_path, parse_dates=["timestamp"]).set_index("timestamp").sort_index()

# Previous_Load: last actual demand point
previous_Load = float(df["ERCOT"].iloc[-1])

# Forecast_Load:
if "Forecast_Load" in df.columns:
    Forecast_Load = float(df["Forecast_Load"].iloc[-1])
else:
    # Proxy: 24h moving average shifted by one step (avoid leakage)
    proxy = df["ERCOT"].rolling(24, min_periods=1).mean().shift(1)
    Forecast_Load = float(proxy.dropna().iloc[-1]) if proxy.dropna().size else float(df["ERCOT"].iloc[-1])

print("Previous_Load:", previous_Load)
print("Forecast_Load:", Forecast_Load)

Previous_Load: 43892.434849
Forecast_Load: 44860.668830125


In [16]:
# Forecast_Load from saved prediction model (with robust fallback)
from pathlib import Path
import numpy as np

try:
    # Try to import TensorFlow/Keras
    from tensorflow import keras

    # Resolve model path
    model_candidates = [
        Path.cwd() / "Save Model" / "model.h5",
        Path.cwd().parent / "Save Model" / "model.h5",
    ]
    for mp in model_candidates:
        if mp.exists():
            model_path = mp
            break
    else:
        raise FileNotFoundError("model.h5 not found in Save Model folder.")

    # Load model
    model = keras.models.load_model(model_path)

    # Infer expected input shape
    in_shape = model.input_shape  # typically (None, time_steps, features)
    # Default sequence length if time_steps is None
    seq_len = in_shape[1] if (isinstance(in_shape, tuple) and len(in_shape) >= 2 and in_shape[1] is not None) else 24
    n_features = in_shape[2] if (isinstance(in_shape, tuple) and len(in_shape) >= 3 and in_shape[2] is not None) else 1

    # Choose feature columns to match n_features (prioritize ERCOT, then weather features)
    preferred_cols = [
        "ERCOT", "tmpc", "relh", "sped", "feel", "p01m",
        "Max-T", "Min-T", "M-label", "W-label", "H-label",
    ]
    available = [c for c in preferred_cols if c in df.columns]
    # If not enough columns, pad with zeros later
    selected_cols = available[:n_features]

    # Build input sequence from the last seq_len rows
    tail = df[selected_cols].tail(seq_len).values if selected_cols else df[["ERCOT"]].tail(seq_len).values
    # Pad/trim to match (seq_len, n_features)
    if tail.shape[0] < seq_len:
        pad_rows = seq_len - tail.shape[0]
        tail = np.vstack([np.zeros((pad_rows, tail.shape[1])), tail])
    if tail.shape[1] < n_features:
        pad_feats = n_features - tail.shape[1]
        tail = np.hstack([tail, np.zeros((seq_len, pad_feats))])
    elif tail.shape[1] > n_features:
        tail = tail[:, :n_features]

    X_last = tail.reshape(1, seq_len, n_features)

    # Predict next load
    y_pred = model.predict(X_last, verbose=0)
    # Extract scalar prediction
    Forecast_Load = float(np.ravel(y_pred)[-1])
    print("✅ Forecast_Load (model):", Forecast_Load)

except Exception as e:
    # Fallback: 24h moving average shifted by one step
    proxy = df["ERCOT"].rolling(24, min_periods=1).mean().shift(1)
    Forecast_Load = float(proxy.dropna().iloc[-1]) if proxy.dropna().size else float(df["ERCOT"].iloc[-1])
    print("⚠️ Model-based forecast unavailable (", str(e), ") -> using proxy Forecast_Load:", Forecast_Load)


⚠️ Model-based forecast unavailable ( No module named 'tensorflow' ) -> using proxy Forecast_Load: 44860.668830125


In [17]:
# Forecast_Load via PyTorch (TorchScript) with robust fallback
from pathlib import Path
import numpy as np
import torch

try:
    # Ensure dataframe is available
    assert 'df' in globals(), "DataFrame 'df' not found; run the data loading cell first."

    # Locate TorchScript model
    ts_candidates = [
        Path.cwd() / "Save Model" / "model.pt",
        Path.cwd().parent / "Save Model" / "model.pt",
    ]
    for mp in ts_candidates:
        if mp.exists():
            model_path = mp
            break
    else:
        raise FileNotFoundError("TorchScript model 'model.pt' not found in Save Model folder.")

    # Load TorchScript model
    model = torch.jit.load(str(model_path))
    model.eval()

    # Build input sequence
    preferred_cols = [
        "ERCOT", "tmpc", "relh", "sped", "feel", "p01m",
        "Max-T", "Min-T", "M-label", "W-label", "H-label",
    ]
    available = [c for c in preferred_cols if c in df.columns]
    selected_cols = available if available else ["ERCOT"]
    seq_len = 24
    n_features = len(selected_cols)

    tail = df[selected_cols].tail(seq_len).to_numpy()
    # Pad rows to seq_len
    if tail.shape[0] < seq_len:
        pad_rows = seq_len - tail.shape[0]
        tail = np.vstack([np.zeros((pad_rows, tail.shape[1]), dtype=tail.dtype), tail])

    # Convert to tensor shape (1, seq_len, n_features)
    x = torch.from_numpy(tail.astype(np.float32)).unsqueeze(0)

    # Forward pass
    with torch.no_grad():
        y = model(x)
    Forecast_Load = float(y.reshape(-1)[-1])
    print("✅ Forecast_Load (PyTorch):", Forecast_Load)

except Exception as e:
    # Fallback: 24h moving average shifted by one step
    proxy = df["ERCOT"].rolling(24, min_periods=1).mean().shift(1)
    Forecast_Load = float(proxy.dropna().iloc[-1]) if proxy.dropna().size else float(df["ERCOT"].iloc[-1])
    print("⚠️ PyTorch-based forecast unavailable (", str(e), ") -> using proxy Forecast_Load:", Forecast_Load)


⚠️ PyTorch-based forecast unavailable ( TorchScript model 'model.pt' not found in Save Model folder. ) -> using proxy Forecast_Load: 44860.668830125
